## Homework 9: Text Classification with Fine-Tuned BERT

In this final homework, we’ll explore **fine-tuning a pre-trained Transformer model (BERT)** for text classification using the **IMDB Movie Review** dataset. You’ll begin with a working baseline notebook and then conduct a series of controlled experiments to understand how data size, context length, and model architecture affect performance.

You’ll complete three problems:

* **Problem 1:** Evaluate how **sequence length** and **learning rate** jointly influence validation loss and generalization.
* **Problem 2:** Measure how **training data size** affects both model performance and total training time.
* **Problem 3:** Compare **two additional models** from the BERT family to analyze the trade-offs between model size and accuracy on this dataset.

In each problem, you’ll report your key metrics, summarize what you observed, and reflect on what you learned.

> **Note:** This homework was developed and tested on **Google Colab**, due to version conflicts when running locally. It is **strongly recommended** that you complete your work on Colab as well.

There are 6 problems, each worth 14 points, and you get one point free if you complete the entire homework.


In [ ]:
# Install once per new Colab runtime
%pip -q uninstall -y pyarrow
%pip -q install -U keras keras-hub tensorflow tensorflow-text datasets evaluate pyarrow

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
import random
import numpy as np
import keras
import keras_hub as kh
import evaluate
from datasets import load_dataset, Dataset, Features, Value, ClassLabel

from keras import mixed_precision                    # generally faster
mixed_precision.set_global_policy("mixed_float16")

### Here is where you can set global hyperparameters for this homework

In [ ]:
# ---------------- Config ----------------
SEED        = 42
MAX_LEN     = 128
EPOCHS      = 3
BATCH       = 32
EVAL_BATCH  = 64
SUBSET_FRAC = 0.25   # <-- 0.25 to train and test on 25% of whole dataset during development;  set to 1.0 for full dataset

keras.utils.set_random_seed(SEED)

### Load and Preprocess the IMDB Movie Review Dataset

In [ ]:
# ---- Load IMDb (raw), join train+test ----
imdb   = load_dataset("imdb")
texts  = list(imdb["train"]["text"]) + list(imdb["test"]["text"])
labels = np.array(list(imdb["train"]["label"]) + list(imdb["test"]["label"]), dtype="int32")

# ---- Build DS with explicit features (label=ClassLabel) ----
features = Features({"text": Value("string"),
                     "label": ClassLabel(num_classes=2, names=["NEG","POS"])})
all_ds = Dataset.from_dict({"text": texts, "label": labels.tolist()}, features=features)

# ---- Optional: take a stratified subset of the FULL dataset ----
if 0.0 < SUBSET_FRAC < 1.0:
    sub = all_ds.train_test_split(train_size=SUBSET_FRAC, seed=SEED, stratify_by_column="label")
    ds_pool = sub["train"]
else:
    ds_pool = all_ds

# ---- Stratified 80/10/10 split on the (possibly smaller) pool ----
# First: 80/20 train+val pool / test
splits = ds_pool.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label")
train_val_pool, test_ds = splits["train"], splits["test"]
# Then: carve 10% of full (i.e., 0.125 of the 80% pool) as validation
splits2 = train_val_pool.train_test_split(test_size=0.125, seed=SEED, stratify_by_column="label")
train_ds, val_ds = splits2["train"], splits2["test"]

# ---- Numpy arrays for Keras fit/predict ----
X_tr = np.array(train_ds["text"], dtype=object); y_tr = np.array(train_ds["label"], dtype="int32")
X_va = np.array(val_ds["text"],   dtype=object); y_va = np.array(val_ds["label"],   dtype="int32")
X_te = np.array(test_ds["text"],  dtype=object); y_te = np.array(test_ds["label"],  dtype="int32")

# ---- Quick summary ----
def _counts(ds):
    arr = np.array(ds["label"], dtype=int)
    return len(arr), np.bincount(arr, minlength=2).tolist()
print(f"Pool after SUBSET_FRAC={SUBSET_FRAC}: {len(ds_pool)} (of {len(all_ds)})")
print("Train:", _counts(train_ds), " Val:", _counts(val_ds), " Test:", _counts(test_ds))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Pool after SUBSET_FRAC=0.25: 12500 (of 50000)
Train: (8750, [4375, 4375])  Val: (1250, [625, 625])  Test: (2500, [1250, 1250])


### Build and train a baseline Distil-Bert Text Classifier

In [ ]:
# ---- Keras Hub preprocessor + classifier ----
preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased", sequence_length=MAX_LEN
)
model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

start = time.time()

# ---- Train with early stopping (restore best val weights) ----
cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1,
)

# ---- Evaluate (accuracy + F1 via `evaluate`) ----
logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
y_pred = logits.argmax(axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")
acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

# Tiny confusion matrix helper (no sklearn needed)
def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

print(f"\nValidation acc (best epoch): {history.history['val_acc'][np.argmin(history.history['val_loss'])]:.3f}")
print(f"\nTest accuracy: {acc:.3f}   Test F1: {f1:.3f}")
print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

end = time.time() - start
print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(end)))

Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 102s 191ms/step - acc: 0.7818 - loss: 0.4527 - val_acc: 0.8392 - val_loss: 0.3449
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - acc: 0.8789 - loss: 0.2898 - val_acc: 0.8592 - val_loss: 0.3400
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step - acc: 0.9159 - loss: 0.2207 - val_acc: 0.8592 - val_loss: 0.3557



Validation acc (best epoch): 0.859

Test accuracy: 0.854   Test F1: 0.850

Confusion matrix:
 [[1093  157]
 [ 209 1041]]

Elapsed time: 00:02:19


# Problem 1 — Mini sweep: context length × learning rate (6 runs)

In this problem we'll see how much **context length** (`MAX_LEN`) helps, and how sensitive fine-tuning is to **learning rate**—without running a huge grid.

## Setup (keep these fixed)

* `SUBSET_FRAC = 0.25`               # use only this percentage of the whole dataset
* `EPOCHS = 3`
* `BATCH = 32` (but see note for 256 below)
* **EarlyStopping** with `restore_best_weights=True`
* Same random `SEED` for all runs
* Same data split for all runs (don’t reshuffle between runs)

### Run these 6 configurations

**For each** `MAX_LEN ∈ {128, 256, 512}`, try **two** learning rates:

* **MAX_LEN = 128**

  * `(LR = 2e-5, BATCH = 32)` – healthy default for shorter contexts.
  * `(LR = 1e-5, BATCH = 32)` – conservative LR; often a touch stabler.

* **MAX_LEN = 256**

  * `(LR = 1e-5, BATCH = 16)` – longer context → lower batch.
  * `(LR = 7.5e-6, BATCH = 16)` – even steadier if loss is noisy.

* **MAX_LEN = 512**  *(heavier quadratic attention cost)*

  * `(LR = 7.5e-6, BATCH = 8)` – safe starting point.
  * `(LR = 5e-6, BATCH = 8)` – extra caution for stability.

**If you hit an Out Of Memory error:**

* At **256** with `BATCH = 16`, drop to `BATCH = 8`.
* At **512** with `BATCH = 8`, drop to `BATCH = 4`.


Then answer the graded questions.


In [ ]:
# Your code here; add as many cells as you need

run_configurations = [
    # MAX_LEN = 128
    {"MAX_LEN": 128, "LR": 2e-5, "BATCH": 32, "description": "MAX_LEN=128, LR=2e-5, BATCH=32"},
    {"MAX_LEN": 128, "LR": 1e-5, "BATCH": 32, "description": "MAX_LEN=128, LR=1e-5, BATCH=32"},
    # MAX_LEN = 256
    {"MAX_LEN": 256, "LR": 1e-5, "BATCH": 16, "description": "MAX_LEN=256, LR=1e-5, BATCH=16"},
    {"MAX_LEN": 256, "LR": 7.5e-6, "BATCH": 16, "description": "MAX_LEN=256, LR=7.5e-6, BATCH=16"},
    # MAX_LEN = 512
    {"MAX_LEN": 512, "LR": 7.5e-6, "BATCH": 8, "description": "MAX_LEN=512, LR=7.5e-6, BATCH=8"},
    {"MAX_LEN": 512, "LR": 5e-6, "BATCH": 8, "description": "MAX_LEN=512, LR=5e-6, BATCH=8"}
]

# Initialize a dictionary to store results from each run
run_dict = {}

for config in run_configurations:
    print("="*80)
    print("Running Scenario: " + config["description"])
    print("="*80)

    current_max_len = config["MAX_LEN"]
    current_lr = config["LR"]
    current_batch = config["BATCH"]

    # ---- Keras Hub preprocessor + classifier ----
    preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
        "distil_bert_base_en_uncased", sequence_length=current_max_len
    )
    model = kh.models.DistilBertTextClassifier.from_preset(
        "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
    )

    model.compile(
        optimizer=keras.optimizers.Adam(current_lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    start = time.time()

    # ---- Train with early stopping (restore best val weights) ----
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
    history = model.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=current_batch,
        callbacks=cb,
        verbose=1,
    )

    # ---- Evaluate (accuracy + F1 via `evaluate`) ----
    logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)

    acc_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")
    acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    f1 = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    # Tiny confusion matrix helper (no sklearn needed)
    def confusion_matrix_np(y_true, y_pred, num_classes=2):
        cm = np.zeros((num_classes, num_classes), dtype=int)
        for t, p in zip(y_true, y_pred):
            cm[t, p] += 1
        return cm

    val_acc_best_epoch = history.history["val_acc"][np.argmin(history.history["val_loss"])]
    print(f"\nValidation acc (best epoch): {val_acc_best_epoch:.3f}")
    print(f"\nTest accuracy: {acc:.3f}   Test F1: {f1:.3f}")
    print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

    end = time.time() - start
    print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(end)))

    # Store results in run_dict
    run_dict[config["description"]] = {
        "MAX_LEN": current_max_len,
        "LR": current_lr,
        "BATCH": current_batch,
        "val_acc_best_epoch": val_acc_best_epoch,
        "test_accuracy": acc,
        "test_f1": f1,
        "elapsed_time": end
    }

Running Scenario: MAX_LEN=128, LR=2e-5, BATCH=32
Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 87s 160ms/step - acc: 0.8010 - loss: 0.4151 - val_acc: 0.8464 - val_loss: 0.3472
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - acc: 0.9018 - loss: 0.2488 - val_acc: 0.8352 - val_loss: 0.3911
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - acc: 0.9358 - loss: 0.1720 - val_acc: 0.8512 - val_loss: 0.3826

Validation acc (best epoch): 0.846

Test accuracy: 0.847   Test F1: 0.855

Confusion matrix:
 [[ 995  255]
 [ 127 1123]]

Elapsed time: 00:01:58
Running Scenario: MAX_LEN=128, LR=1e-5, BATCH=32
Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 87s 158ms/step - acc: 0.7814 - loss: 0.4546 - val_acc: 0.8520 - val_loss: 0.3437
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - acc: 0.8800 - loss: 0.2907 - val_acc: 0.8560 - val_loss: 0.3474
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - acc: 0.9153 - loss: 0.2203 - val_acc: 0.8608 - val_loss: 0.3540

Validation acc (best epoch): 0.852

Test a

In [ ]:
print("\nProblem 1 Results:")
for i in run_dict:
  print(run_dict[i])


Problem 1 Results:
{'MAX_LEN': 128, 'LR': 2e-05, 'BATCH': 32, 'val_acc_best_epoch': 0.8464000225067139, 'test_accuracy': 0.8472, 'test_f1': 0.8546423135464232, 'elapsed_time': 118.35982537269592}
{'MAX_LEN': 128, 'LR': 1e-05, 'BATCH': 32, 'val_acc_best_epoch': 0.8519999980926514, 'test_accuracy': 0.8444, 'test_f1': 0.8430818878580073, 'elapsed_time': 118.25182437896729}
{'MAX_LEN': 256, 'LR': 1e-05, 'BATCH': 16, 'val_acc_best_epoch': 0.902400016784668, 'test_accuracy': 0.8868, 'test_f1': 0.8899260987942434, 'elapsed_time': 162.4813311100006}
{'MAX_LEN': 256, 'LR': 7.5e-06, 'BATCH': 16, 'val_acc_best_epoch': 0.9047999978065491, 'test_accuracy': 0.8908, 'test_f1': 0.8922226608764311, 'elapsed_time': 137.695814371109}
{'MAX_LEN': 512, 'LR': 7.5e-06, 'BATCH': 8, 'val_acc_best_epoch': 0.9136000275611877, 'test_accuracy': 0.9152, 'test_f1': 0.9156050955414012, 'elapsed_time': 206.73500204086304}
{'MAX_LEN': 512, 'LR': 5e-06, 'BATCH': 8, 'val_acc_best_epoch': 0.9192000031471252, 'test_accura

### Graded Questions

In [ ]:
# Set a1a to the validation accuracy at min validation loss for your best configuration found in this problem

a1a = 0.9192           # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1a = {a1a:.4f}')

a1a = 0.9192


#### Question a1b:

* Does **more context** (128 → 256 → 512) consistently help?
* How much effect did the learning rate have on the validation accuracy?


#### Your Answer Here:

More context certainly helped as there was consistenly higher accuracy and F1 scores. However, there was a huge improvement between length of 128 and 256 and only marginal improvement between 256 and 512. It was interesting to see not much overfitting as the gap in validation and test accuracy was relatively even across scenarios. The learning rate effects were marginal with improvements below 0.01. if anything there was so mild overfitting with different learning rates.

## Problem 2 — How much data is enough?

In this problem, you’ll investigate how model performance scales with dataset size.

**Setup.**
Use the best `MAX_LEN` and `LR` values you found in **Problem 1**.

**What to do:**

1. For each value of `SUBSET_FRAC ∈ {0.25, 0.50, 0.75, 1.00}`, train your model once and observe the displayed performance metrics.
2. Answer the discussion question below.




In [ ]:
# Your code here; add as many cells as you need

BEST_MAX_LEN = 512
BEST_LR = 7.5e-06
SUBSET_FRACS = [0.25, 0.50, 0.75, 1.00]

problem2_results = {}

for subset_frac in SUBSET_FRACS:
  print("="*80)
  print(f"Running with SUBSET_FRAC={{subset_frac}}")
  print("="*80)

  # ---- Load IMDb (raw), join train+test ----
  # This block is re-executed to re-create the dataset with the current SUBSET_FRAC
  imdb   = load_dataset("imdb")
  texts  = list(imdb["train"]["text"]) + list(imdb["test"]["text"])
  labels = np.array(list(imdb["train"]["label"]) + list(imdb["test"]["label"]), dtype="int32")

  # ---- Build DS with explicit features (label=ClassLabel) ----
  features = Features({"text": Value("string"),
                       "label": ClassLabel(num_classes=2, names=["NEG","POS"])})
  all_ds = Dataset.from_dict({"text": texts, "label": labels.tolist()}, features=features)

  # ---- Optional: take a stratified subset of the FULL dataset ----
  if 0.0 < subset_frac < 1.0:
      sub = all_ds.train_test_split(train_size=subset_frac, seed=SEED, stratify_by_column="label")
      ds_pool = sub["train"]
  else:
      ds_pool = all_ds

  # ---- Stratified 80/10/10 split on the (possibly smaller) pool ----
  # First: 80/20 train+val pool / test
  splits = ds_pool.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label")
  train_val_pool, test_ds = splits["train"], splits["test"]
  # Then: carve 10% of full (i.e., 0.125 of the 80% pool) as validation
  splits2 = train_val_pool.train_test_split(test_size=0.125, seed=SEED, stratify_by_column="label")
  train_ds, val_ds = splits2["train"], splits2["test"]

  # ---- Numpy arrays for Keras fit/predict ----
  X_tr = np.array(train_ds["text"], dtype=object); y_tr = np.array(train_ds["label"], dtype="int32")
  X_va = np.array(val_ds["text"],   dtype=object); y_va = np.array(val_ds["label"],   dtype="int32")
  X_te = np.array(test_ds["text"],  dtype=object); y_te = np.array(test_ds["label"],  dtype="int32")

  # ---- Quick summary ----
  def _counts(ds):
      arr = np.array(ds["label"], dtype=int)
      return len(arr), np.bincount(arr, minlength=2).tolist()
  print(f"Pool after SUBSET_FRAC={{subset_frac}}: {len(ds_pool)} (of {len(all_ds)})")
  print("Train:", _counts(train_ds), " Val:", _counts(val_ds), " Test:", _counts(test_ds))

  # ---- Keras Hub preprocessor + classifier ----
  preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
      "distil_bert_base_en_uncased", sequence_length=BEST_MAX_LEN
  )
  model = kh.models.DistilBertTextClassifier.from_preset(
      "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
  )

  model.compile(
      optimizer=keras.optimizers.Adam(BEST_LR),
      loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
      metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
  )

  start = time.time()

  # ---- Train with early stopping (restore best val weights) ----
  cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
  history = model.fit(
      X_tr,
      y_tr,
      validation_data=(X_va, y_va),
      epochs=EPOCHS,
      batch_size=BATCH, # Using global BATCH for now, assuming it's appropriate for BEST_MAX_LEN
      callbacks=cb,
      verbose=1,
  )

  # ---- Evaluate (accuracy + F1 via `evaluate`) ----
  logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
  y_pred = logits.argmax(axis=-1)

  acc_metric = evaluate.load("accuracy")
  f1_metric = evaluate.load("f1")
  acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
  f1 = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

  # Tiny confusion matrix helper (no sklearn needed)
  def confusion_matrix_np(y_true, y_pred, num_classes=2):
      cm = np.zeros((num_classes, num_classes), dtype=int)
      for t, p in zip(y_true, y_pred):
          cm[t, p] += 1
      return cm

  val_acc_best_epoch = history.history["val_acc"][np.argmin(history.history["val_loss"])]
  print(f"\nValidation acc (best epoch): {val_acc_best_epoch:.3f}")
  print(f"\nTest accuracy: {acc:.3f}   Test F1: {f1:.3f}")
  print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

  end = time.time() - start
  print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(end)))

  # Store results in problem2_results
  problem2_results[f"SUBSET_FRAC={subset_frac}"] = {
      "SUBSET_FRAC": subset_frac,
      "val_acc_best_epoch": val_acc_best_epoch,
      "test_accuracy": acc,
      "test_f1": f1,
      "elapsed_time": end
  }

print("\nProblem 2 Results:")
print(problem2_results)


Running with SUBSET_FRAC={subset_frac}
Pool after SUBSET_FRAC={subset_frac}: 12500 (of 50000)
Train: (8750, [4375, 4375])  Val: (1250, [625, 625])  Test: (2500, [1250, 1250])
Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 119s 239ms/step - acc: 0.8200 - loss: 0.3936 - val_acc: 0.9048 - val_loss: 0.2419
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 21s 77ms/step - acc: 0.9201 - loss: 0.2057 - val_acc: 0.9120 - val_loss: 0.2299
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 21s 77ms/step - acc: 0.9447 - loss: 0.1515 - val_acc: 0.9168 - val_loss: 0.2242

Validation acc (best epoch): 0.917

Test accuracy: 0.909   Test F1: 0.909

Confusion matrix:
 [[1144  106]
 [ 121 1129]]

Elapsed time: 00:02:55
Running with SUBSET_FRAC={subset_frac}
Pool after SUBSET_FRAC={subset_frac}: 25000 (of 50000)
Train: (17500, [8750, 8750])  Val: (2500, [1250, 1250])  Test: (5000, [2500, 2500])
Epoch 1/3
547/547 ━━━━━━━━━━━━━━━━━━━━ 130s 156ms/step - acc: 0.8595 - loss: 0.3227 - val_acc: 0.9148 - val_loss: 0.2061
Epoch 2/3
547/547 ━━━

Pool after SUBSET_FRAC={subset_frac}: 37500 (of 50000)
Train: (26250, [13125, 13125])  Val: (3750, [1875, 1875])  Test: (7500, [3750, 3750])
Epoch 1/3
821/821 ━━━━━━━━━━━━━━━━━━━━ 148s 125ms/step - acc: 0.8771 - loss: 0.2922 - val_acc: 0.9219 - val_loss: 0.2055
Epoch 2/3
821/821 ━━━━━━━━━━━━━━━━━━━━ 59s 72ms/step - acc: 0.9286 - loss: 0.1867 - val_acc: 0.9253 - val_loss: 0.2058
Epoch 3/3
821/821 ━━━━━━━━━━━━━━━━━━━━ 59s 72ms/step - acc: 0.9474 - loss: 0.1477 - val_acc: 0.9211 - val_loss: 0.2240

Validation acc (best epoch): 0.922

Test accuracy: 0.918   Test F1: 0.917

Confusion matrix:
 [[3494  256]
 [ 356 3394]]

Elapsed time: 00:04:47
Running with SUBSET_FRAC={subset_frac}
Pool after SUBSET_FRAC={subset_frac}: 50000 (of 50000)
Train: (35000, [17500, 17500])  Val: (5000, [2500, 2500])  Test: (10000, [5000, 5000])
Epoch 1/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 167s 113ms/step - acc: 0.8854 - loss: 0.2757 - val_acc: 0.9324 - val_loss: 0.1813
Epoch 2/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 78s 71ms/

### Graded Questions

In [ ]:
# Set a2a to the validation accuracy at min validation loss for your best configuration found in this problem
# (Yes, it is probably at 1.0!)

a2a = 0.934            # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a2a = {a2a:.4f}')

a2a = 0.9340


#### Question a2b:

Summarize what you observed as dataset size increased. Given that validation metrics are typically reliable to only about two decimal places, do the performance gains justify using the entire dataset? What trade-offs between accuracy and computation time did you notice?

#### Your Answer Here:

Validation accuracy increased from 0.917 to 0.934. run times stesdily increased from 2.5 minutes to 6 minutes. Given the size of this dataset and the modest accuracy gain it was worth the trade off. On a much larger dataset with significnalty longer training time, or with inferior hardware, this may not be worth it.

# Problem 3 — Model swap: speed vs. accuracy (why: capacity matters)

In this problem we will compare encoder-only backbones of different sizes.

**Setup.** Keep the best `MAX_LEN`, `LR`, and `SUBSET_FRAC` from Problems 1–2. Only change the model/preset:

* **DistilBERT** (current baseline)
* **BERT-base** (larger/usually stronger)

**How to switch (two lines each).**

* DistilBERT:

  ```python
  preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset("distil_bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.DistilBertTextClassifier.from_preset("distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

* BERT-base:

  ```python
  preproc = kh.models.BertTextClassifierPreprocessor.from_preset("bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.BertTextClassifier.from_preset("bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

**What to do.**

1. Train/evaluate each model once with identical settings.
2. Observe the performance metrics for each.
3. Answer the graded questions.



In [ ]:
# Your code here; add as many cells as you wish

BEST_MAX_LEN = 512
BEST_LR = 7.5e-06
SUBSET_FRAC = 1 # Using 0.25 as per Problem 1 setup, assuming it's still intended to be used

models = {
    "DistilBERT": "distil_bert_base_en_uncased",
    "BERT-base": "bert_base_en_uncased"
}

problem3_results = {}

# --- Re-prepare the dataset based on the chosen SUBSET_FRAC for problem 3 ---
# This block is re-executed to re-create the dataset with the current SUBSET_FRAC
imdb   = load_dataset("imdb")
texts  = list(imdb["train"]["text"]) + list(imdb["test"]["text"])
labels = np.array(list(imdb["train"]["label"]) + list(imdb["test"]["label"]), dtype="int32")

features = Features({"text": Value("string"),
                     "label": ClassLabel(num_classes=2, names=["NEG","POS"])})
all_ds = Dataset.from_dict({"text": texts, "label": labels.tolist()}, features=features)

if 0.0 < SUBSET_FRAC < 1.0:
    sub = all_ds.train_test_split(train_size=SUBSET_FRAC, seed=SEED, stratify_by_column="label")
    ds_pool = sub["train"]
else:
    ds_pool = all_ds

splits = ds_pool.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label")
train_val_pool, test_ds = splits["train"], splits["test"]
splits2 = train_val_pool.train_test_split(test_size=0.125, seed=SEED, stratify_by_column="label")
train_ds, val_ds = splits2["train"], splits2["test"]

X_tr = np.array(train_ds["text"], dtype=object); y_tr = np.array(train_ds["label"], dtype="int32")
X_va = np.array(val_ds["text"],   dtype=object); y_va = np.array(val_ds["label"],   dtype="int32")
X_te = np.array(test_ds["text"],  dtype=object); y_te = np.array(test_ds["label"],  dtype="int32")

print(f"Using SUBSET_FRAC={SUBSET_FRAC} for Problem 3")


for model_name_alias, preset_name in models.items():
  print("="*80)
  print(f"Running with model: {model_name_alias} ({preset_name})")
  print("="*80)

  # Determine preprocessor and model class based on the preset_name
  if "distil_bert" in preset_name:
    preprocessor_class = kh.models.DistilBertTextClassifierPreprocessor
    model_class = kh.models.DistilBertTextClassifier
  elif "bert_base" in preset_name:
    preprocessor_class = kh.models.BertTextClassifierPreprocessor
    model_class = kh.models.BertTextClassifier
  else:
    raise ValueError(f"Unknown model preset: {preset_name}")

  # ---- Keras Hub preprocessor + classifier ----
  preproc = preprocessor_class.from_preset(
      preset_name, sequence_length=BEST_MAX_LEN
  )
  model = model_class.from_preset(
      preset_name, num_classes=2, preprocessor=preproc
  )

  model.compile(
      optimizer=keras.optimizers.Adam(BEST_LR),
      loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
      metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
  )

  start = time.time()

  # ---- Train with early stopping (restore best val weights) ----
  cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
  history = model.fit(
      X_tr,
      y_tr,
      validation_data=(X_va, y_va),
      epochs=EPOCHS,
      batch_size=BATCH, # Using global BATCH
      callbacks=cb,
      verbose=1,
  )

  # ---- Evaluate (accuracy + F1 via `evaluate`) ----
  logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
  y_pred = logits.argmax(axis=-1)

  acc_metric = evaluate.load("accuracy")
  f1_metric = evaluate.load("f1")
  acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
  f1 = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

  # Tiny confusion matrix helper (no sklearn needed)
  def confusion_matrix_np(y_true, y_pred, num_classes=2):
      cm = np.zeros((num_classes, num_classes), dtype=int)
      for t, p in zip(y_true, y_pred):
          cm[t, p] += 1
      return cm

  val_acc_best_epoch = history.history["val_acc"][np.argmin(history.history["val_loss"])]
  print(f"\nValidation acc (best epoch): {val_acc_best_epoch:.3f}")
  print(f"\nTest accuracy: {acc:.3f}   Test F1: {f1:.3f}")
  print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

  end = time.time() - start
  print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(end)))

  # Store results
  problem3_results[model_name_alias] = {
      "preset_name": preset_name,
      "val_acc_best_epoch": val_acc_best_epoch,
      "test_accuracy": acc,
      "test_f1": f1,
      "elapsed_time": end
  }

print("\nProblem 3 Results:")
print(problem3_results)


Using SUBSET_FRAC=1 for Problem 3
Running with model: DistilBERT (distil_bert_base_en_uncased)
Epoch 1/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 158s 104ms/step - acc: 0.8833 - loss: 0.2791 - val_acc: 0.9308 - val_loss: 0.1831
Epoch 2/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 79s 72ms/step - acc: 0.9311 - loss: 0.1824 - val_acc: 0.9340 - val_loss: 0.1753
Epoch 3/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 79s 72ms/step - acc: 0.9501 - loss: 0.1388 - val_acc: 0.9340 - val_loss: 0.1778

Validation acc (best epoch): 0.934

Test accuracy: 0.927   Test F1: 0.927

Confusion matrix:
 [[4648  352]
 [ 373 4627]]

Elapsed time: 00:05:34
Running with model: BERT-base (bert_base_en_uncased)
Epoch 1/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 284s 189ms/step - acc: 0.9025 - loss: 0.2472 - val_acc: 0.9392 - val_loss: 0.1628
Epoch 2/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 153s 140ms/step - acc: 0.9447 - loss: 0.1524 - val_acc: 0.9416 - val_loss: 0.1592
Epoch 3/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 152s 139ms/step - acc: 0.9666 - loss: 0.1017 - val_acc

### Graded Questions

In [ ]:
# Set a1a to the validation accuracy at min validation loss for your best model found in this problem

a3a = 0.942       # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a3a = {a3a:.4f}')

a3a = 0.9420


#### Question a3b:

**Answer briefly.**

* Which model gives the best **accuracy/F1**?
* Which is **fastest** per epoch?
* Given limited development time or compute resources, which model is the best **overall choice** and why?

#### Your Answer Here:

Bert-base had stronger accuracy and F1. BERT-base took approximately twice as long as distilBERT. It is clearly a tradeoff we would need to make on a larger dataset. Accuracy increased modestly from 0.934 to 0.942. Improvements were similar on the test scores with accuracy improving from 0.928 to 0.934 and F1 from 0.927 to 0.935. On my hardware the cylce time increased from 5.5 minutes to 10 minutes. This is immaterial in nominal terms. However, on much larger datasets that may require hours or days it is not worth the slim accuracy improvement.